# 第7章：位置编码——让模型理解顺序

> **预计学习时间：约 30 分钟**
>
> 上一章我们搭建了完整的 Transformer 架构。但有一个关键问题还没解决：注意力机制本身**不区分词的先后顺序**。本章将带你理解位置编码的原理，亲手实现正弦位置编码，并了解现代大模型使用的 RoPE 技术。

**运行环境**: Kaggle Notebook (CPU 即可)
**前置要求**: 完成第6章 Transformer 架构全景

---
## 0. 环境准备

In [ ]:
# =============================================
# 0. 导入依赖
# =============================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

# 设置随机种子，确保每次运行结果一致
torch.manual_seed(42)
np.random.seed(42)

plt.rcParams['font.size'] = 12

print(f"PyTorch 版本: {torch.__version__}")
print("本章主要使用 CPU，无需 GPU")

---
## 1. 为什么需要位置编码？

### 类比：打乱顺序的扑克牌

想象你面前有 5 张扑克牌，按顺序排列：A、2、3、4、5。现在有人打乱了顺序变成 3、5、A、4、2。

对于人来说，你一眼就能看出"这些牌被打乱了"。但对于注意力机制来说，**它根本不在乎顺序**。

为什么？回忆第5章的注意力计算：每个词和所有其他词算点积，得到注意力权重。这个计算过程是**置换不变的（permutation invariant）**——把输入序列打乱顺序，每个词得到的注意力加权结果完全一样。

这意味着，对于纯注意力机制：

| 句子 | 含义 | 注意力结果 |
|------|------|-----------|
| "狗 咬 人" | 狗咬了人 | **完全相同!** |
| "人 咬 狗" | 人咬了狗 | **完全相同!** |

这显然是不对的！语言的含义很大程度上取决于词的**顺序**。

### 解决方案：给每个位置一个"门牌号"

解决办法很简单：给序列中每个位置一个独特的向量（位置编码），加到词嵌入上。这样即使两个词相同，在不同位置时，模型看到的输入就不同了。

```
没有位置编码：
  "猫" → [0.2, 0.5, 0.1]   （无论在第1个位置还是第5个位置，都一样）

有位置编码：
  位置0的"猫" → [0.2, 0.5, 0.1] + [1.0, 0.0, 0.5] = [1.2, 0.5, 0.6]
  位置3的"猫" → [0.2, 0.5, 0.1] + [0.1, 0.9, -0.3] = [0.3, 1.4, -0.2]
  → 不同位置的同一个词，变成了不同的向量！
```

![为什么需要位置编码](images/why_position_encoding.png)

*图1：没有位置编码时，注意力机制无法区分词的顺序*

> 💡 **记忆要点**
> - 注意力机制是**位置无关**的——打乱顺序不影响结果
> - 位置编码 = 给每个位置一个独特的向量，加到词嵌入上
> - 公式：最终输入 = 词嵌入（Word Embedding）+ 位置编码（Positional Encoding）

In [ ]:
# =============================================
# 1.1 实验验证：注意力机制真的不区分位置吗？
# =============================================
# 我们用一个简单的 Self-Attention 层来证明这一点

d_model = 16  # 用小维度便于观察

# 创建一个简单的 Self-Attention（没有位置编码）
W_Q = nn.Linear(d_model, d_model, bias=False)
W_K = nn.Linear(d_model, d_model, bias=False)
W_V = nn.Linear(d_model, d_model, bias=False)

def simple_attention(x):
    """计算 Self-Attention（不含位置编码）"""
    Q = W_Q(x)  # [batch, seq_len, d_model]
    K = W_K(x)
    V = W_V(x)
    # Scaled Dot-Product Attention
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_model)
    weights = F.softmax(scores, dim=-1)
    output = weights @ V
    return output

# 创建一个序列：4 个不同的词向量
torch.manual_seed(42)
word_a = torch.randn(1, d_model)  # "狗"
word_b = torch.randn(1, d_model)  # "咬"
word_c = torch.randn(1, d_model)  # "人"
word_d = torch.randn(1, d_model)  # "了"

# 原始顺序: "狗 咬 人 了"
original = torch.stack([word_a, word_b, word_c, word_d], dim=1)  # [1, 4, d_model]

# 打乱顺序: "人 了 狗 咬"
shuffled = torch.stack([word_c, word_d, word_a, word_b], dim=1)  # [1, 4, d_model]

# 分别计算注意力
with torch.no_grad():
    out_original = simple_attention(original)  # [1, 4, d_model]
    out_shuffled = simple_attention(shuffled)  # [1, 4, d_model]

# 比较："狗" 在原始序列中是第0个位置的输出
# 在打乱序列中，"狗" 在第2个位置
# 如果注意力不区分位置，两个输出应该完全相同
dog_output_original = out_original[0, 0]  # "狗" 在原始中的输出
dog_output_shuffled = out_shuffled[0, 2]  # "狗" 在打乱中的输出

diff = (dog_output_original - dog_output_shuffled).abs().max().item()

print("实验：验证注意力机制的位置无关性")
print("=" * 50)
print(f"原始顺序: 狗(0) 咬(1) 人(2) 了(3)")
print(f"打乱顺序: 人(0) 了(1) 狗(2) 咬(3)")
print()
print(f'"狗" 在原始序列中的输出（位置0）:')
print(f'  {dog_output_original[:6].tolist()}...')
print(f'"狗" 在打乱序列中的输出（位置2）:')
print(f'  {dog_output_shuffled[:6].tolist()}...')
print(f'\n两个输出的最大差异: {diff:.10f}')
print()
if diff < 1e-6:
    print("结论: 差异几乎为零！注意力机制完全不区分位置。")
    print("      '狗咬人' 和 '人咬狗' 在纯注意力看来是一样的！")
    print("      → 这就是为什么我们需要位置编码")
else:
    print(f"注意: 存在微小的数值差异 ({diff:.2e})，这是浮点精度导致的")

---
## 2. 正弦位置编码（Sinusoidal Positional Encoding）

### 原始 Transformer 的方案

2017 年的 "Attention Is All You Need" 论文提出了一种优雅的位置编码方法：用不同频率的正弦和余弦函数来编码每个位置。

**公式：**

| 维度（偶数） | 维度（奇数） |
|---|---|
| PE(pos, 2i) = sin(pos / 10000^(2i/d_model)) | PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model)) |

其中 `pos` 是位置（0, 1, 2, ...），`i` 是维度索引（0, 1, 2, ..., d_model/2 - 1）。

### 直觉理解：像时钟一样编码位置

想象一排时钟，每个时钟的转速不同：

- **第一个时钟**：秒针（转得最快）—— 每一秒转一格
- **第二个时钟**：分针（中等速度）—— 每60秒转一格
- **第三个时钟**：时针（转得最慢）—— 每3600秒转一格

如果你同时看这三个时钟的指针位置，就能精确地知道**当前的时间**。

正弦位置编码的思路完全一样：
- **低维度**用高频波（变化快，区分相邻位置）
- **高维度**用低频波（变化慢，区分远距离位置）

每个位置的编码 = 一组不同频率的正弦/余弦值，组合起来**唯一确定**这个位置。

![正弦位置编码](images/sinusoidal_encoding.png)

*图2：不同维度的正弦编码就像不同速度的时钟指针——组合起来唯一确定每个位置*

### 为什么选择正弦/余弦？

1. **值域有界**：始终在 [-1, 1] 之间，不会数值爆炸
2. **相对位置可表示**：PE(pos+k) 可以用 PE(pos) 的线性变换得到（三角恒等式）
3. **可泛化到未见过的长度**：训练时用长度 512，推理时用长度 1024 也没问题

> 💡 **记忆要点**
> - 正弦位置编码 = 不同频率的正弦波叠加
> - 低维度 = 高频（区分相邻位置），高维度 = 低频（区分远距离）
> - 优势：值有界、可表示相对位置、可泛化到更长序列

In [ ]:
# =============================================
# 2.1 从零实现正弦位置编码
# =============================================

class SinusoidalPositionalEncoding(nn.Module):
    """正弦位置编码
    
    给序列中的每个位置生成一个独特的编码向量，
    使用不同频率的 sin/cos 函数。
    
    公式: PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
          PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        
        # 创建位置编码矩阵 [max_len, d_model]
        pe = torch.zeros(max_len, d_model)
        
        # pos: 位置索引 [max_len, 1]
        # 例如 max_len=100 → pos = [[0], [1], [2], ..., [99]]
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        
        # div_term: 频率缩放因子
        # 2i/d_model 从 0 到 1，10000^(2i/d_model) 从 1 到 10000
        # → 频率从高到低
        # 这里用 exp + log 来计算，等价于 10000^(2i/d_model)，但数值更稳定
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        # div_term 的形状: [d_model/2]
        # div_term[0] = 1.0（最高频），div_term[-1] ≈ 0.0001（最低频）
        
        # 偶数维度用 sin，奇数维度用 cos
        pe[:, 0::2] = torch.sin(pos * div_term)  # 所有偶数列
        pe[:, 1::2] = torch.cos(pos * div_term)  # 所有奇数列
        
        # 增加 batch 维度: [max_len, d_model] → [1, max_len, d_model]
        pe = pe.unsqueeze(0)
        
        # register_buffer: 把 pe 注册为模型的一部分（保存/加载时会包含），
        # 但不作为可训练参数（不参与梯度计算）
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """
        x: [batch_size, seq_len, d_model]
        返回: x + 位置编码（只取前 seq_len 个位置）
        """
        seq_len = x.size(1)
        # self.pe[:, :seq_len] → [1, seq_len, d_model]
        # 自动广播到 batch_size
        return x + self.pe[:, :seq_len]


# 测试
d_model = 64
max_len = 100

pe_module = SinusoidalPositionalEncoding(d_model, max_len)

# 模拟输入：5 个词，每个 64 维
x = torch.zeros(1, 5, d_model)  # 全零输入，方便观察纯位置编码
output = pe_module(x)

print("正弦位置编码测试:")
print(f"  输入形状:  {list(x.shape)}")
print(f"  输出形状:  {list(output.shape)}")
print(f"  位置编码表形状: {list(pe_module.pe.shape)}  [1, max_len={max_len}, d_model={d_model}]")
print()
print("位置 0 的编码（前 8 维）:")
print(f"  {output[0, 0, :8].tolist()}")
print("位置 1 的编码（前 8 维）:")
print(f"  {output[0, 1, :8].tolist()}")
print("位置 2 的编码（前 8 维）:")
print(f"  {output[0, 2, :8].tolist()}")
print()
print("观察: 每个位置的编码都不同，前面几维变化快（高频），后面几维变化慢（低频）")

In [ ]:
# =============================================
# 2.2 可视化正弦位置编码
# =============================================
# 通过 4 张子图，从不同角度观察正弦位置编码的特点

d_model = 128
max_len = 100
pe_module = SinusoidalPositionalEncoding(d_model, max_len)
pe_matrix = pe_module.pe[0].numpy()  # [max_len, d_model]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- 左上: 整体热力图 ---
# 每一行是一个位置的编码向量，每一列是一个维度
# 颜色越红=值越大(接近+1)，越蓝=值越小(接近-1)
im = axes[0, 0].imshow(pe_matrix[:50, :64], cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
axes[0, 0].set_xlabel('Dimension (i)', fontsize=12)
axes[0, 0].set_ylabel('Position (pos)', fontsize=12)
axes[0, 0].set_title('Sinusoidal Position Encoding Matrix', fontsize=14)
plt.colorbar(im, ax=axes[0, 0], label='Encoding Value')

# --- 右上: 不同维度的波形 ---
# 挑选 5 个维度，展示从高频到低频的变化
# dim 0 变化最快（能区分相邻位置），dim 126 变化最慢（能区分远距离位置）
positions = np.arange(50)
for dim_idx, color, label in [
    (0, '#e74c3c', 'Dim 0 (highest freq)'),
    (10, '#f39c12', 'Dim 10'),
    (30, '#2ecc71', 'Dim 30'),
    (60, '#3498db', 'Dim 60'),
    (126, '#9b59b6', 'Dim 126 (lowest freq)')
]:
    axes[0, 1].plot(positions, pe_matrix[:50, dim_idx], color=color, linewidth=2, label=label)
axes[0, 1].set_xlabel('Position', fontsize=12)
axes[0, 1].set_ylabel('Encoding Value', fontsize=12)
axes[0, 1].set_title('Different Dimensions = Different Frequencies', fontsize=14)
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# --- 左下: 位置之间的相似度矩阵 ---
# 用余弦相似度衡量：两个位置编码向量的"方向"有多接近
# 对角线 = 1（自身相似度），离对角线越远 = 相似度越低
pe_tensor = torch.tensor(pe_matrix[:50])  # [50, d_model]
# 归一化后做矩阵乘法 = 余弦相似度
pe_norm = F.normalize(pe_tensor, dim=-1)
similarity = (pe_norm @ pe_norm.T).numpy()  # [50, 50]

im2 = axes[1, 0].imshow(similarity, cmap='RdYlBu_r', aspect='auto', vmin=-0.5, vmax=1.0)
axes[1, 0].set_xlabel('Position', fontsize=12)
axes[1, 0].set_ylabel('Position', fontsize=12)
axes[1, 0].set_title('Position Similarity (Cosine)', fontsize=14)
plt.colorbar(im2, ax=axes[1, 0], label='Cosine Similarity')

# --- 右下: 选几个参考位置，看它们和其他位置的相似度曲线 ---
# 每条曲线在自身位置处 = 1，随着距离增大逐渐下降
ref_positions = [0, 10, 25]
colors = ['#e74c3c', '#3498db', '#2ecc71']
for ref_pos, color in zip(ref_positions, colors):
    axes[1, 1].plot(range(50), similarity[ref_pos], color=color, 
                     linewidth=2, label=f'Similarity to pos {ref_pos}')
axes[1, 1].set_xlabel('Position', fontsize=12)
axes[1, 1].set_ylabel('Cosine Similarity', fontsize=12)
axes[1, 1].set_title('Similarity to Reference Positions', fontsize=14)
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察:")
print("  左上: 低维度(左侧)变化快，高维度(右侧)变化慢 — 不同频率的波")
print("  右上: 不同维度的波形，从高频到低频")
print("  左下: 相邻位置相似度高（对角线附近偏红），远距离位置相似度低")
print("  右下: 每个位置和自己最相似(=1)，随距离增大逐渐下降")

---
## 3. 可学习位置编码（Learned Positional Encoding）

### GPT-2 和 BERT 的选择

正弦位置编码虽然优雅，但 GPT-2 和 BERT 实际上用了一种更简单的方法：**直接学习**每个位置的编码。

原理非常简单：创建一个 `nn.Embedding(max_len, d_model)` 查找表，位置 0、1、2、... 各对应一个可训练的向量。训练过程中，模型自动学习什么样的位置编码最有用。

```
正弦位置编码（固定）:       可学习位置编码（训练）:
  pos 0 → sin/cos 计算得到     pos 0 → 从查找表中查出（随机初始化，训练后更新）
  pos 1 → sin/cos 计算得到     pos 1 → 从查找表中查出
  pos 2 → sin/cos 计算得到     pos 2 → 从查找表中查出
  ...                          ...
  可以外推到任意长度             最长只能到 max_len（GPT-2 是 1024）
```

### 两种方法的对比

| 特性 | 正弦位置编码 | 可学习位置编码 |
|------|------------|-------------|
| 参数量 | 0（无需训练） | max_len × d_model |
| 灵活性 | 固定模式 | 可适应具体任务 |
| 长度外推 | 天然支持任意长度 | 超过 max_len 就不行 |
| 代表模型 | 原始 Transformer | GPT-2, BERT |
| 实际效果 | 略差 | 略好（因为更灵活） |

![可学习 vs 固定位置编码](images/learned_vs_fixed.png)

*图3：正弦（固定）vs 可学习位置编码的对比*

> 💡 **记忆要点**
> - 可学习位置编码 = `nn.Embedding(max_len, d_model)` 查找表
> - 优势：更灵活，效果略好
> - 劣势：有最大长度限制，超过就失效
> - GPT-2 的最大长度是 1024 个 token

In [ ]:
# =============================================
# 3.1 实现可学习位置编码
# =============================================

class LearnedPositionalEncoding(nn.Module):
    """可学习位置编码（GPT-2 / BERT 使用的方式）
    
    用 nn.Embedding 创建一个查找表，每个位置对应一个可训练的向量。
    训练过程中，模型自动学习最优的位置表示。
    """
    def __init__(self, d_model, max_len=512):
        super().__init__()
        # nn.Embedding(max_len, d_model):
        #   一个 max_len × d_model 的可训练矩阵
        #   输入位置索引 → 输出对应的 d_model 维向量
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.max_len = max_len
    
    def forward(self, x):
        """
        x: [batch_size, seq_len, d_model]
        """
        seq_len = x.size(1)
        
        # 生成位置索引: [0, 1, 2, ..., seq_len-1]
        # .to(x.device) 确保在同一设备（CPU/GPU）上
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)  # [1, seq_len]
        
        # 查表得到位置编码: [1, seq_len] → [1, seq_len, d_model]
        pos_encoding = self.position_embedding(positions)
        
        return x + pos_encoding


# 对比两种位置编码
d_model = 64
max_len = 100

sinusoidal_pe = SinusoidalPositionalEncoding(d_model, max_len)
learned_pe = LearnedPositionalEncoding(d_model, max_len)

# 统计参数量
sin_params = sum(p.numel() for p in sinusoidal_pe.parameters())
learn_params = sum(p.numel() for p in learned_pe.parameters())

print("两种位置编码对比:")
print(f"  正弦位置编码 - 可训练参数: {sin_params}（全部是固定的 buffer，不需要训练）")
print(f"  可学习位置编码 - 可训练参数: {learn_params:,}（= max_len × d_model = {max_len} × {d_model}）")
print()

# 验证可学习位置编码是随机初始化的
print("可学习位置编码（初始化后，未训练）:")
x = torch.zeros(1, 5, d_model)
with torch.no_grad():
    out_learned = learned_pe(x)
print(f"  位置 0 编码（前 8 维）: {out_learned[0, 0, :8].tolist()}")
print(f"  位置 1 编码（前 8 维）: {out_learned[0, 1, :8].tolist()}")
print("  → 随机初始化的值，训练后会变成有意义的位置表示")

In [ ]:
# =============================================
# 3.2 可视化对比两种位置编码
# =============================================
# 左图：正弦 PE 有明显的波纹规律（数学公式决定）
# 右图：可学习 PE 在训练前是随机的（训练后会出现规律）

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 正弦位置编码 ---
# 从预计算的 buffer 中取出前 50 个位置、前 64 个维度
sin_pe = sinusoidal_pe.pe[0, :50, :64].numpy()
im1 = axes[0].imshow(sin_pe, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
axes[0].set_xlabel('Dimension', fontsize=12)
axes[0].set_ylabel('Position', fontsize=12)
axes[0].set_title('Sinusoidal PE (Fixed, No Training)', fontsize=14)
plt.colorbar(im1, ax=axes[0], label='Value')

# --- 可学习位置编码（未训练） ---
# 直接从 nn.Embedding 的权重矩阵中取值
# 此时是 PyTorch 默认的随机初始化（标准正态分布）
with torch.no_grad():
    learn_pe = learned_pe.position_embedding.weight[:50, :64].numpy()
im2 = axes[1].imshow(learn_pe, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
axes[1].set_xlabel('Dimension', fontsize=12)
axes[1].set_ylabel('Position', fontsize=12)
axes[1].set_title('Learned PE (Random Init, Before Training)', fontsize=14)
plt.colorbar(im2, ax=axes[1], label='Value')

plt.tight_layout()
plt.show()

print("对比:")
print("  左图(正弦): 有明显的规律性波纹 — 这是数学公式定义的固定模式")
print("  右图(可学习): 看起来像随机噪声 — 因为还没开始训练")
print("  训练后，右图会呈现出有规律的模式（模型学到了什么位置信息有用）")

---
## 4. RoPE：旋转位置编码

### 现代大模型的选择

正弦位置编码和可学习位置编码都属于**绝对位置编码**——它们给每个位置一个固定的编码。但语言理解中，**相对位置**往往比绝对位置更重要。

比如在句子"我昨天去了北京"中，"昨天"和"去了"的关系主要取决于它们**相距1个位置**，而不是它们分别在第2和第3个位置。

RoPE（Rotary Position Embedding，旋转位置编码）是 2021 年提出的方法，被 LLaMA、DeepSeek、Qwen 等现代大模型广泛采用。

### 核心思想：用旋转编码相对位置

RoPE 的核心思想非常优雅：

1. 把向量的每两个维度看作一个 2D 平面上的点
2. 根据位置，把这个点**旋转**一个角度
3. 位置 m 的向量旋转 m × θ 度，位置 n 的向量旋转 n × θ 度
4. 计算注意力时，两个向量的点积**只取决于它们的相对位置 (m-n)**

这就好比：

> 想象一个操场上有很多人，每个人面朝的方向取决于他的位置编号。位置 0 朝北，位置 1 朝北偏东 10°，位置 2 朝北偏东 20°......
> 
> 两个人能不能"对上眼"（点积大小），只取决于他们面朝方向的**夹角差**——也就是他们之间的**相对距离**。

![RoPE 旋转概念](images/rope_concept.png)

*图4：RoPE 用旋转角度编码位置——两个向量的点积只取决于相对位置差*

### 为什么 RoPE 更好？

| 特性 | 绝对位置编码 | RoPE |
|------|-----------|------|
| 编码方式 | 加到输入上 | 应用在 Q 和 K 上 |
| 位置信息 | 绝对位置 | 相对位置 |
| 长度外推 | 差（可学习）/ 一般（正弦） | 好（通过调整频率基数可外推） |
| 代表模型 | GPT-2, BERT | LLaMA, DeepSeek, Qwen |

> 💡 **记忆要点**
> - RoPE = 把 Q、K 向量根据位置旋转不同角度
> - 点积只取决于相对位置差 → 天然捕获相对位置关系
> - 现代主流大模型（LLaMA、DeepSeek、Qwen）都使用 RoPE

In [ ]:
# =============================================
# 4.1 RoPE 的简化演示（2D 旋转）
# =============================================
# RoPE 的核心操作是 2D 旋转。在实际模型中，d_model 维的向量被拆成
# d_model/2 组 2D 子空间，每组独立旋转。这里用最简单的单组 2D 来演示原理。

def rotate_2d(vector, angle):
    """将 2D 向量旋转指定角度
    
    旋转矩阵: [[cos(θ), -sin(θ)],
               [sin(θ),  cos(θ)]]
    
    Args:
        vector: [2] 的向量（Q 或 K 的一个 2D 子空间）
        angle: 旋转角度（弧度），= 位置 × 基础频率
    Returns:
        旋转后的 [2] 向量
    """
    cos_a = math.cos(angle)
    sin_a = math.sin(angle)
    x, y = vector[0].item(), vector[1].item()
    # 标准的 2D 旋转公式
    return torch.tensor([x * cos_a - y * sin_a, x * sin_a + y * cos_a])


# =============================================
# 核心思想演示：
#   位置 m 的向量旋转 m*θ，位置 n 的向量旋转 n*θ
#   它们的点积只取决于 (m-n)*θ —— 即相对位置!
# =============================================

theta = 0.3  # 每个位置旋转的基础角度（实际模型中有多个不同频率）

# 假设 Q 和 K 都是 2D 向量（简化示例）
q_original = torch.tensor([1.0, 0.5])  # 原始 Q 向量
k_original = torch.tensor([0.8, 0.3])  # 原始 K 向量

print("RoPE 2D 旋转演示")
print("=" * 60)
print(f"原始 Q: {q_original.tolist()}")
print(f"原始 K: {k_original.tolist()}")
print(f"基础旋转角度 θ = {theta:.1f} 弧度")
print()

# 测试不同的绝对位置，但保持相对距离不变（都是 3）
# 如果 RoPE 正确编码了相对位置，这些点积应该完全相同
print(f"{'Q位置':>6} {'K位置':>6} {'相对距离':>8} {'点积':>10} {'变化':>8}")
print("-" * 50)

# 基准：Q 在位置 0，K 在位置 3（相对距离=3）
q_pos, k_pos = 0, 3
q_rotated = rotate_2d(q_original, q_pos * theta)  # 旋转 0×0.3 = 0 弧度
k_rotated = rotate_2d(k_original, k_pos * theta)  # 旋转 3×0.3 = 0.9 弧度
base_dot = (q_rotated @ k_rotated).item()
print(f"{q_pos:>6} {k_pos:>6} {k_pos-q_pos:>8} {base_dot:>10.4f} {'(baseline)':>8}")

# 改变绝对位置，但保持相对距离=3
# 点积应该保持不变（验证 RoPE 的相对位置性质）
for q_pos, k_pos in [(5, 8), (10, 13), (100, 103)]:
    q_rotated = rotate_2d(q_original, q_pos * theta)
    k_rotated = rotate_2d(k_original, k_pos * theta)
    dot = (q_rotated @ k_rotated).item()
    diff = abs(dot - base_dot)
    print(f"{q_pos:>6} {k_pos:>6} {k_pos-q_pos:>8} {dot:>10.4f} {f'Δ={diff:.6f}':>8}")

print()
print("关键发现:")
print("  不管绝对位置怎么变，只要相对距离相同(=3)，点积就完全相同！")
print("  → RoPE 天然编码了相对位置信息")
print()

# 对比：不同相对距离的点积变化
# 这个变化规律就是模型用来感知"两个词距离多远"的信号
print("不同相对距离下的点积变化:")
print(f"{'相对距离':>8} {'点积':>10}")
print("-" * 20)
q_pos = 0
for gap in [0, 1, 2, 3, 5, 10, 20]:
    k_pos = q_pos + gap
    q_rotated = rotate_2d(q_original, q_pos * theta)
    k_rotated = rotate_2d(k_original, k_pos * theta)
    dot = (q_rotated @ k_rotated).item()
    print(f"{gap:>8} {dot:>10.4f}")
print()
print("观察: 点积随相对距离变化 → 模型可以利用这个信号来理解位置关系")

In [ ]:
# =============================================
# 4.2 可视化 RoPE 旋转效果
# =============================================
# 左图：展示同一个向量在不同位置被旋转到不同方向
# 右图：展示 Q·K 点积随相对距离的变化——RoPE 的核心信号

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- 左图：不同位置的向量旋转 ---
theta = 0.5  # 基础旋转角度
q = torch.tensor([1.0, 0.0])  # 单位向量，方便观察纯旋转效果

# 位置 0~7 的向量被旋转 0*θ, 1*θ, 2*θ, ... 7*θ
colors = plt.cm.viridis(np.linspace(0, 1, 8))
for pos in range(8):
    rotated = rotate_2d(q, pos * theta)
    # 画从原点到旋转后位置的箭头
    axes[0].annotate('', xy=(rotated[0].item(), rotated[1].item()), xytext=(0, 0),
                      arrowprops=dict(arrowstyle='->', color=colors[pos], lw=2))
    axes[0].plot(rotated[0].item(), rotated[1].item(), 'o', color=colors[pos], markersize=8)
    # 标注位置编号
    axes[0].annotate(f'pos={pos}', 
                      xy=(rotated[0].item()*1.15, rotated[1].item()*1.15),
                      fontsize=10, color=colors[pos], fontweight='bold')

# 画单位圆参考线（所有旋转后的向量都在圆上）
circle = plt.Circle((0, 0), 1, fill=False, color='gray', linestyle='--', alpha=0.5)
axes[0].add_patch(circle)
axes[0].set_xlim(-1.5, 1.5)
axes[0].set_ylim(-1.5, 1.5)
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel('Dimension 0', fontsize=12)
axes[0].set_ylabel('Dimension 1', fontsize=12)
axes[0].set_title('RoPE: Vector Rotation by Position', fontsize=14)
axes[0].axhline(y=0, color='k', linewidth=0.5)
axes[0].axvline(x=0, color='k', linewidth=0.5)

# --- 右图：相对位置 vs 点积 ---
# 固定 Q 在位置 0，移动 K 从位置 0 到 30
# 观察点积如何随相对距离变化（这就是模型感知位置的信号）
q_original = torch.tensor([1.0, 0.5])
k_original = torch.tensor([0.8, 0.3])
theta = 0.3

gaps = list(range(31))
dots = []
for gap in gaps:
    q_rotated = rotate_2d(q_original, 0 * theta)      # Q 在位置 0
    k_rotated = rotate_2d(k_original, gap * theta)     # K 在位置 gap
    dots.append((q_rotated @ k_rotated).item())

axes[1].plot(gaps, dots, 'o-', color='#3498db', linewidth=2, markersize=6)
axes[1].set_xlabel('Relative Position (gap)', fontsize=12)
axes[1].set_ylabel('Dot Product (Q · K)', fontsize=12)
axes[1].set_title('RoPE: Dot Product vs Relative Position', fontsize=14)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("左图: 每个位置的向量被旋转不同角度 — 位置越大，旋转越多")
print("右图: Q·K 的值只取决于相对距离 — 这就是 RoPE 编码相对位置的方式")

---
## 5. 对比实验：有无位置编码的效果

让我们实际验证位置编码的作用。我们创建一个小型 Transformer，分别在有和没有位置编码的情况下，看模型能否区分不同顺序的序列。

In [ ]:
# =============================================
# 5.1 实验：位置编码对注意力模式的影响
# =============================================
# 实验设计:
#   1. 创建一个序列（6 个词向量），以原始顺序和打乱顺序输入
#   2. 分别在"无位置编码"和"有位置编码"条件下计算注意力
#   3. 比较：打乱顺序后注意力权重变化了多少
#   预期结果：无 PE 时变化极小，有 PE 时变化显著

# 重用第6章的 MultiHeadAttention（标准多头注意力实现）
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # 每个头的维度
        # Q, K, V 的线性投影层
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)  # 输出投影
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        # 投影并拆分成多个头: [batch, seq, d_model] → [batch, n_heads, seq, d_k]
        Q = self.W_Q(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        # Scaled Dot-Product Attention
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = F.softmax(scores, dim=-1)  # 注意力权重 [batch, n_heads, seq, seq]
        # 加权求和并合并多头
        context = (attn_weights @ V).transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_O(context), attn_weights


d_model = 32
n_heads = 4
seq_len = 6

# 共享同一个注意力模块（确保公平对比）
torch.manual_seed(42)
attn = MultiHeadAttention(d_model, n_heads)

# 创建 6 个不同的词向量（模拟一个 6 词的句子）
torch.manual_seed(0)
words = torch.randn(seq_len, d_model)

# 定义两种词序：原始顺序和打乱顺序
original_order = torch.arange(seq_len)
shuffled_order = torch.tensor([3, 0, 5, 2, 4, 1])

x_original = words[original_order].unsqueeze(0)  # [1, 6, d_model]
x_shuffled = words[shuffled_order].unsqueeze(0)   # [1, 6, d_model]

# --- 实验组1：不加位置编码 ---
with torch.no_grad():
    _, weights_no_pe_orig = attn(x_original)
    _, weights_no_pe_shuf = attn(x_shuffled)

# --- 实验组2：加上正弦位置编码 ---
pe = SinusoidalPositionalEncoding(d_model, max_len=100)
with torch.no_grad():
    x_orig_with_pe = pe(x_original)   # 词嵌入 + 位置编码
    x_shuf_with_pe = pe(x_shuffled)   # 词嵌入 + 位置编码（位置不同!）
    _, weights_with_pe_orig = attn(x_orig_with_pe)
    _, weights_with_pe_shuf = attn(x_shuf_with_pe)

# 计算打乱前后的注意力权重差异（只看第一个头）
diff_no_pe = (weights_no_pe_orig[0, 0] - weights_no_pe_shuf[0, 0]).abs().mean().item()
diff_with_pe = (weights_with_pe_orig[0, 0] - weights_with_pe_shuf[0, 0]).abs().mean().item()

print("实验：位置编码对注意力的影响")
print("=" * 50)
print(f"原始顺序: {original_order.tolist()}")
print(f"打乱顺序: {shuffled_order.tolist()}")
print()
print(f"无位置编码 — 注意力权重差异: {diff_no_pe:.6f}")
print(f"有位置编码 — 注意力权重差异: {diff_with_pe:.6f}")
print()
if diff_with_pe > diff_no_pe:
    ratio = diff_with_pe / max(diff_no_pe, 1e-10)
    print(f"结论: 加入位置编码后，打乱顺序导致的注意力差异增大了 {ratio:.0f} 倍！")
    print("      → 位置编码让模型能够区分不同的词序")
else:
    print("注意: 结果可能因随机种子而异")

In [ ]:
# =============================================
# 5.2 可视化对比：有无位置编码的注意力模式
# =============================================
# 4 张热力图对比：
#   上排 = 无 PE（打乱前后几乎一样 → 模型不区分位置）
#   下排 = 有 PE（打乱后明显不同 → 模型感知到了位置变化）

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 用 w0, w1, ... 标注每个位置对应的词
labels_orig = [f'w{i}' for i in original_order.tolist()]
labels_shuf = [f'w{i}' for i in shuffled_order.tolist()]

# 4 组注意力权重矩阵（都取第一个头, head=0）
matrices = [
    (weights_no_pe_orig[0, 0].detach().numpy(), labels_orig, 'No PE: Original Order'),
    (weights_no_pe_shuf[0, 0].detach().numpy(), labels_shuf, 'No PE: Shuffled Order'),
    (weights_with_pe_orig[0, 0].detach().numpy(), labels_orig, 'With PE: Original Order'),
    (weights_with_pe_shuf[0, 0].detach().numpy(), labels_shuf, 'With PE: Shuffled Order'),
]

for ax, (w, labels, title) in zip(axes.flat, matrices):
    # 绘制注意力权重热力图，颜色越深 = 注意力越高
    im = ax.imshow(w, cmap='Blues', aspect='auto', vmin=0, vmax=0.5)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_yticklabels(labels, fontsize=11)
    ax.set_xlabel('Key', fontsize=11)
    ax.set_ylabel('Query', fontsize=11)
    ax.set_title(title, fontsize=13)
    # 在每个格子中标注具体数值，方便精确对比
    for i in range(len(labels)):
        for j in range(len(labels)):
            color = 'white' if w[i, j] > 0.3 else 'black'
            ax.text(j, i, f'{w[i,j]:.2f}', ha='center', va='center', fontsize=9, color=color)

plt.tight_layout()
plt.show()

print("对比分析:")
print("  上排(无 PE): 打乱顺序后，同一个词(如 w0)的注意力模式几乎不变")
print("              → 模型根本不知道词的位置变了！")
print("  下排(有 PE): 打乱顺序后，注意力模式明显改变")
print("              → 位置编码让模型能感知到词序的变化")

---
## 6. 位置编码的演进总结

![位置编码发展历程](images/position_encoding_summary.png)

*图5：位置编码技术的演进——从正弦编码到 RoPE*

### 三代位置编码

| 代际 | 方法 | 年份 | 代表模型 | 核心特点 |
|------|------|------|---------|---------|
| 第一代 | 正弦位置编码 | 2017 | Transformer | 固定公式，无需训练，可外推 |
| 第二代 | 可学习位置编码 | 2018 | GPT-2, BERT | 查找表，更灵活，有长度限制 |
| 第三代 | RoPE 旋转位置编码 | 2021 | LLaMA, DeepSeek | 编码相对位置，外推能力强 |

### 5 个关键要点

**1. 为什么需要位置编码** — 注意力机制是位置无关的（置换不变），必须额外注入位置信息

**2. 正弦位置编码** — 用不同频率的 sin/cos 编码位置，低维高频、高维低频，类比多个时钟

**3. 可学习位置编码** — `nn.Embedding` 查找表，灵活但有最大长度限制，GPT-2/BERT 使用

**4. RoPE** — 通过旋转 Q/K 向量编码位置，点积只取决于相对位置，现代大模型的首选

**5. 实际效果** — 有无位置编码会显著影响注意力模式，模型需要位置信息才能理解语序

### 下一章预告

在 **第8章：阶段总结——构建迷你 Transformer** 中，我们将：
- 把前 7 章学到的所有组件组装起来
- 从零实现一个完整的小型 Transformer 模型（1M 参数）
- 用它完成简单的序列补全任务
- 看到模型从随机输出到学会基本模式的完整训练过程

---

> **参考资料**
> - Vaswani et al., 2017: *Attention Is All You Need* — 正弦位置编码
> - Devlin et al., 2019: *BERT* — 可学习位置编码
> - Su et al., 2021: *RoFormer: Enhanced Transformer with Rotary Position Embedding* — RoPE 原始论文
> - Kazemnejad et al., 2023: *The Impact of Positional Encoding on Length Generalization* — 位置编码对长度泛化的影响